# Xây dựng Custom DenseNet-121 cho Oxford-IIIT Pet từ đầu (Data-Centric Foreground Edition)

**Yêu cầu xây dựng model:**
- Input: 224x224x3
- Classes: 37 giống loài (12 giống mèo, 25 giống chó) thuộc 2 loài động vật Chó và Mèo
- Framework: PyTorch
- Pretrained: Không sử dụng (From Scratch)
> **Lưu ý Data-Centric:** Notebook này được trang bị tiền xử lý Foreground Extraction (cắt chặt BBox vật thể từ Trimap) và Background Neutralization (khử nền xám trung tính $) để bảo đảm đối sánh công bằng 100% với PetNet-Hybrid V3.


In [ ]:
# [Môi trường Kaggle / Colab] Tự động tải mã nguồn từ GitHub và chuyển vào đúng thư mục làm việc
import os
import subprocess

if os.path.exists("/kaggle") or os.path.exists("/content"):
    REPO_URL = "https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git"
    TARGET_DIR = "/kaggle/working/ml-semester-cat-dog-classifier" if os.path.exists("/kaggle") else "/content/ml-semester-cat-dog-classifier"
    
    if not os.path.exists(TARGET_DIR):
        print(f">> Đang tải repository từ {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, TARGET_DIR], check=True)
        print(f">> Tải thành công repository vào {TARGET_DIR}")
    else:
        print(f">> Repository đã tồn tại tại {TARGET_DIR}")
        try:
            subprocess.run(["git", "-C", TARGET_DIR, "pull"], check=False)
        except Exception:
            pass
        
    # Chuyển thư mục làm việc về đúng thư mục notebook của Phước
    notebook_dir = os.path.join(TARGET_DIR, "notebooks", "Phuoc", "06_densenet121")
    if os.path.exists(notebook_dir):
        os.chdir(notebook_dir)
    else:
        os.chdir(TARGET_DIR)
    print(f">> Thư mục làm việc hiện tại: {os.getcwd()}")
else:
    print(f">> Chạy môi trường Local: {os.getcwd()}")


## Bước 1: Input các thư viện

In [ ]:
import os
import sys
import json
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

# Thiết lập Random Seed chuẩn xác
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Tự động tìm thư mục gốc chứa package src (tương thích cả folder mới 3 cấp, Kaggle, Colab)
current_path = Path.cwd()
possible_roots = [
    current_path,
    current_path.parent,
    current_path.parent.parent,
    current_path.parent.parent.parent,
    Path("/kaggle/working/ml-semester-cat-dog-classifier"),
    Path("/content/ml-semester-cat-dog-classifier")
]
project_root = None
for r in possible_roots:
    if (r / "src").is_dir() and (r / "src" / "dataset.py").exists():
        project_root = r.resolve()
        break

if project_root is not None:
    for p in [str(project_root), str(project_root / "src")]:
        if p not in sys.path:
            sys.path.insert(0, p)
    print(f">> Đã nhận diện Project Root: {project_root}")

# Nạp modules: import chuẩn từ package src
try:
    from src.dataset import OxfordPetForegroundDataset, get_transforms, CLAHETransform
except ImportError:
    from dataset import OxfordPetForegroundDataset, get_transforms, CLAHETransform

# Nhận diện phần cứng: Tự động phát hiện CUDA (Kaggle T4), MPS (Apple Silicon), hoặc CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Thiết bị đang sử dụng: {torch.cuda.get_device_name(0)} (CUDA GPU) | Seed: {SEED}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f"Thiết bị đang sử dụng: Apple Silicon MPS | Seed: {SEED}")
else:
    device = torch.device('cpu')
    print(f"Thiết bị đang sử dụng: CPU | Seed: {SEED}")


## Bước 2: Argparse & Thiết lập siêu tham số

In [ ]:
def find_dataset_path():
    """Tự động tìm kiếm thư mục Oxford-IIIT Pet trên Kaggle (/kaggle/input), Colab, hoặc Local."""
    data_dir = None
    
    # 1. Tìm trên Kaggle /kaggle/input
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if "images" in dirs and os.path.exists(os.path.join(root, "annotations", "trainval.txt")):
                has_trimaps = os.path.exists(os.path.join(root, "annotations", "trimaps"))
                lower_p = root.lower()
                if has_trimaps or ("oxford" in lower_p and "new" not in lower_p):
                    data_dir = root
                    break
                elif data_dir is None:
                    data_dir = root
    
    # 2. Tìm trên Local / Colab
    if data_dir is None:
        candidates = [
            "../../../dataset/oxford-iiit-pet",
            "../../dataset/oxford-iiit-pet",
            str(project_root / "dataset" / "oxford-iiit-pet") if project_root else "",
            "/content/data/oxford-iiit-pet",
            "/content/oxford-iiit-pet",
            "dataset/oxford-iiit-pet"
        ]
        for c in candidates:
            if c and os.path.exists(os.path.join(c, "annotations", "trainval.txt")):
                data_dir = c
                break
                
    # 3. Tự động tải qua torchvision nếu trên cloud chưa có
    if data_dir is None and (os.path.exists("/kaggle") or os.path.exists("/content")):
        print(">> Đang tự động nạp Oxford-IIIT Pet qua torchvision...")
        import torchvision
        dl_root = "/kaggle/working/data" if os.path.exists("/kaggle") else "/content/data"
        torchvision.datasets.OxfordIIITPet(root=dl_root, split="trainval", download=True)
        data_dir = os.path.join(dl_root, "oxford-iiit-pet")
        
    return data_dir

def parse_args():
    parser = argparse.ArgumentParser(description="Custom DenseNet-121 (From Scratch on Oxford-IIIT Pet)")
    dataset_path = find_dataset_path()
    parser.add_argument("--data_dir", type=str, default=dataset_path, help="Đường dẫn bộ dữ liệu Oxford-IIIT Pet")
    parser.add_argument("--task", type=str, default="breed", choices=["breed", "binary"])
    parser.add_argument("--img_size", type=int, default=224, help="Kích thước ảnh chuẩn 224x224")
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size 64 tối ưu trên GPU T4 (hoặc 32 trên máy nhỏ)")
    parser.add_argument("--epochs", type=int, default=100, help="Số epochs huấn luyện 100")
    parser.add_argument("--lr", type=float, default=5e-4, help="Learning rate tinh chỉnh 0.0005 cho AdamW")
    parser.add_argument("--weight_decay", type=float, default=1e-2, help="Decoupled weight decay 0.01 chống overfitting")
    parser.add_argument("--label_smoothing", type=float, default=0.1, help="Label smoothing 0.1 tăng khả năng tổng quát hóa")
    parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")
    parser.add_argument("--plots_dir", type=str, default="./plots")
    parser.add_argument("--crop_foreground", type=bool, default=True, help="Cắt chặt Bounding Box vùng thú cưng")
    parser.add_argument("--remove_background", type=bool, default=True, help="Khử nền xám trung tính")
    try:
        args, _ = parser.parse_known_args()
    except:
        args = parser.parse_args([])
    return args

args = parse_args()
print(f"Đường dẫn Dataset: {args.data_dir}")
print(f"Chế độ Data-Centric: Crop Foreground = {args.crop_foreground} | Neutral Gray BG = {args.remove_background}")


## Bước 3: Load Data & Data-Centric Foreground Extraction

Để đảm bảo tính đối sánh công bằng chính xác với kiến trúc PetNet-Hybrid V3, mô hình này được áp dụng cùng cơ chế **Data-Centric AI**:
1. **Trimap BBox Crop:** Tự động cắt chặt theo hộp bao của toàn bộ cơ thể và khuôn mặt thú cưng, loại bỏ các viền cảnh nền trống.
2. **Background Neutralization:** Khử toàn bộ các pixel cảnh nền (sofa, bãi cỏ, sàn nhà) thành màu xám trung tính $ nhằm triệt tiêu gradient nhiễu ngoại cảnh.
3. **Augmentation:** Duy trì đầy đủ CLAHE LAB và Part Crop Augmentation để trích xuất hoa văn lông và tai/mắt.


In [ ]:
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader

try:
    from src.dataset import OxfordPetForegroundDataset, get_transforms, CLAHETransform
except ImportError:
    from dataset import OxfordPetForegroundDataset, get_transforms, CLAHETransform

# Data Augmentation nâng cao: Extreme Part Crop (8% - 40% diện tích) + CLAHE LAB
train_transform = get_transforms(img_size=args.img_size, is_train=True, use_part_aug=True, use_clahe=True)
test_transform = get_transforms(img_size=args.img_size, is_train=False)

trainset = OxfordPetForegroundDataset(
    root_dir=args.data_dir, split="trainval", task=args.task,
    transform=train_transform, crop_foreground=args.crop_foreground, remove_background=args.remove_background
)
trainloader = DataLoader(
    trainset, batch_size=args.batch_size, shuffle=True,
    num_workers=2, pin_memory=(device.type == "cuda")
)

testset = OxfordPetForegroundDataset(
    root_dir=args.data_dir, split="test", task=args.task,
    transform=test_transform, crop_foreground=args.crop_foreground, remove_background=args.remove_background
)
testloader = DataLoader(
    testset, batch_size=args.batch_size, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda")
)

classes = trainset.class_names if args.task == "breed" else ["Cat", "Dog"]
num_classes = len(classes)
print(f"Tổng số mẫu Train: {len(trainset):,} | Test: {len(testset):,} | Số lớp: {num_classes}")
print(f"Batch Size: {args.batch_size} | Kích thước ảnh: {args.img_size}x{args.img_size} px")

# Trích xuất và trực quan hóa bộ phận chính xác kết hợp Trimap (tách nền) + XML Ground Truth
import numpy as np
import xml.etree.ElementTree as ET
from PIL import ImageDraw

def extract_parts_trimap_xml(img_path, annotations_dir, target_size=(224, 224)):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    xml_path = os.path.join(annotations_dir, "xmls", f"{base_name}.xml")
    trimap_path = os.path.join(annotations_dir, "trimaps", f"{base_name}.png")
    
    # 1. Trích xuất Head BBox từ XML nếu có
    head_box = None
    if os.path.exists(xml_path):
        try:
            tree = ET.parse(xml_path)
            bndbox = tree.getroot().find("object").find("bndbox")
            xmin = max(0, int(bndbox.find("xmin").text))
            ymin = max(0, int(bndbox.find("ymin").text))
            xmax = min(w, int(bndbox.find("xmax").text))
            ymax = min(h, int(bndbox.find("ymax").text))
            head_box = (xmin, ymin, xmax, ymax)
        except Exception:
            pass

    # 2. Trích xuất Mask cơ thể chó/mèo từ Trimap (1 = pet body, 3 = contour) để loại bỏ 100% nền nhà
    body_box = None
    if os.path.exists(trimap_path):
        try:
            tri_arr = np.array(Image.open(trimap_path))
            pet_mask = (tri_arr == 1) | (tri_arr == 3)
            
            if head_box is not None:
                head_mid_y = int((head_box[1] + head_box[3]) / 2)
                lower_body_mask = pet_mask.copy()
                lower_body_mask[:head_mid_y, :] = False
                
                indices = np.argwhere(lower_body_mask)
                if len(indices) > 50:
                    b_ymin, b_xmin = indices.min(axis=0)
                    b_ymax, b_xmax = indices.max(axis=0)
                    body_box = (int(b_xmin), int(b_ymin), int(b_xmax), int(b_ymax))
            
            if body_box is None:
                indices = np.argwhere(pet_mask)
                if len(indices) > 50:
                    ymin_f, xmin_f = indices.min(axis=0)
                    ymax_f, xmax_f = indices.max(axis=0)
                    body_box = (int(xmin_f), int(ymin_f + 0.4 * (ymax_f - ymin_f)), int(xmax_f), int(ymax_f))
                    if head_box is None:
                        head_box = (int(xmin_f), int(ymin_f), int(xmax_f), int(ymin_f + 0.6 * (ymax_f - ymin_f)))
        except Exception:
            pass

    # 3. Fallback hình học nếu thiếu cả XML lẫn Trimap
    if head_box is None:
        head_box = (0, 0, w, int(h * 0.55))
    if body_box is None:
        body_box = (0, int(h * 0.40), w, h)

    head_crop = img.crop(head_box).resize(target_size, Image.Resampling.BILINEAR)
    body_crop = img.crop(body_box).resize(target_size, Image.Resampling.BILINEAR)

    # Đánh dấu Bounding Box trực quan trên ảnh gốc
    annotated_img = img.copy()
    draw = ImageDraw.Draw(annotated_img)
    draw.rectangle(head_box, outline="#06d6a0", width=4)   # Khung xanh: Đầu / Tai
    draw.rectangle(body_box, outline="#ef476f", width=4)   # Khung đỏ: Chân / Thân (Trimap sạch nền)

    return annotated_img, head_crop, body_crop

# Minh họa trực quan trên ảnh mẫu
try:
    sample_idx = 10
    sample_path, _, _, _, sample_name = trainset.samples[sample_idx]
    ann_img, crop_head, crop_paw = extract_parts_trimap_xml(sample_path, os.path.join(args.data_dir, "annotations"), target_size=(args.img_size, args.img_size))
    clahe_img = CLAHETransform(clip_limit=3.0, p=1.0)(Image.open(sample_path).convert("RGB"))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(ann_img); axes[0].set_title(f"Ảnh gốc: {sample_name} | Xanh: Đầu, Đỏ: Chân", fontsize=10); axes[0].axis("off")
    axes[1].imshow(crop_head); axes[1].set_title("Bộ phận: Tai / Đầu (Head)", fontsize=11, fontweight="bold"); axes[1].axis("off")
    axes[2].imshow(crop_paw); axes[2].set_title("Bộ phận: Chân / Thân (Paws/Body)", fontsize=11, fontweight="bold"); axes[2].axis("off")
    axes[3].imshow(clahe_img); axes[3].set_title("Lọc CLAHE Tăng Nét Lông", fontsize=11); axes[3].axis("off")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f">> Bỏ qua hiển thị ảnh minh họa ({e}).")


## Bước 4: Kiến trúc Model From Scratch

In [ ]:
class _DenseLayer(nn.Module):
    def __init__(self, in_features, growth_rate, bn_size=4):
        super(_DenseLayer, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_features)
        self.conv1 = nn.Conv2d(in_features, bn_size * growth_rate, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(bn_size * growth_rate)
        self.conv2 = nn.Conv2d(bn_size * growth_rate, growth_rate, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(torch.relu(self.bn1(x)))
        out = self.conv2(torch.relu(self.bn2(out)))
        return torch.cat([x, out], 1)

class _DenseBlock(nn.Module):
    def __init__(self, num_layers, in_features, growth_rate, bn_size=4):
        super(_DenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            layers.append(_DenseLayer(in_features + i * growth_rate, growth_rate, bn_size))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class _Transition(nn.Module):
    def __init__(self, in_features, out_features):
        super(_Transition, self).__init__()
        self.bn = nn.BatchNorm2d(in_features)
        self.conv = nn.Conv2d(in_features, out_features, kernel_size=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        out = self.conv(torch.relu(self.bn(x)))
        return self.pool(out)

class CustomDenseNet(nn.Module):
    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16), num_init_features=64, num_classes=37):
        super(CustomDenseNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, num_init_features, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            block = _DenseBlock(num_layers=num_layers, in_features=num_features, growth_rate=growth_rate)
            self.features.add_module(f'denseblock{i + 1}', block)
            num_features = num_features + num_layers * growth_rate
            if i != len(block_config) - 1:
                trans = _Transition(in_features=num_features, out_features=num_features // 2)
                self.features.add_module(f'transition{i + 1}', trans)
                num_features = num_features // 2

        self.features.add_module('norm5', nn.BatchNorm2d(num_features))
        self.classifier = nn.Linear(num_features, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.features(x)
        out = torch.relu(features)
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        return self.classifier(out)

model = CustomDenseNet(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Đã chọn mô hình: Custom DenseNet-121 (From Scratch)")
print(f"Khởi tạo trọng số Kaiming He Normal hoàn tất!")
print(f"Tổng số tham số: {total_params:,}")

## Bước 5: Huấn luyện (Training & Checkpoints)

In [ ]:
os.makedirs(args.checkpoint_dir, exist_ok=True)
criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

best_ckpt = os.path.join(args.checkpoint_dir, "densenet121_foreground_best.pth")
start_epoch = 1
best_acc = 0.0
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Đọc lại checkpoint nếu đã có từ trước (Resume feature)
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    else:
        scheduler.last_epoch = ckpt["epoch"]
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt.get("best_acc", 0.0)
    h = ckpt.get("history", {})
    train_losses = h.get("train_loss", [])
    test_losses = h.get("test_loss", [])
    train_accs = h.get("train_acc", [])
    test_accs = h.get("test_acc", [])
    print(f"Đã nạp checkpoint tốt nhất! Tiếp tục từ Epoch {start_epoch:02d} đến {args.epochs:02d} (Best Acc: {best_acc:.2f}%)")

print(f"Bắt đầu huấn luyện Custom DenseNet-121 từ Epoch {start_epoch:02d} đến {args.epochs:02d}...")

for epoch in range(start_epoch, args.epochs + 1):
    # Train 1 epoch
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets, _ in trainloader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # Test
    model.eval()
    test_loss_sum, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ in testloader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss_sum += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            test_correct += preds.eq(targets).sum().item()
            test_total += targets.size(0)

    test_loss = test_loss_sum / test_total
    test_acc = 100.0 * test_correct / test_total
    scheduler.step()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d} | LR: {current_lr:.6f} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%", flush=True)

    if test_acc > best_acc or epoch == 1:
        if test_acc > best_acc:
            best_acc = test_acc
        ckpt_data = {
            "epoch": epoch,
            "model_name": "densenet121",
            "total_params": total_params,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_acc": best_acc,
            "history": {
                "train_loss": train_losses, "test_loss": test_losses,
                "train_acc": train_accs, "test_acc": test_accs
            }
        }
        torch.save(ckpt_data, best_ckpt)

print(f"Huấn luyện hoàn tất! Best Test Acc đạt được: {best_acc:.2f}%")


## Bước 6: Trực quan hóa & Xuất kết quả (Plots & Metrics)

In [ ]:
import torch.nn.functional as F
os.makedirs(args.plots_dir, exist_ok=True)
epochs_range = range(1, len(train_losses) + 1)

# 1. Đường cong Loss & Accuracy
try:
    plt.figure(figsize=(14, 5))

    # Subplot 1: Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, 'o-', label='Train Loss', color='#2b5c8f', lw=2, markersize=3)
    plt.plot(epochs_range, test_losses, 's-', label='Test Loss', color='#e76f51', lw=2, markersize=3)
    plt.title("Loss Curves - Custom DenseNet-121", fontweight="bold")
    plt.xlabel('Epoch')
    plt.ylabel('Loss (Label Smoothing = 0.1)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    # Subplot 2: Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, train_accs, 'o-', label='Train Acc', color='#2a9d8f', lw=2, markersize=3)
    plt.plot(epochs_range, test_accs, 's-', label='Test Acc', color='#e76f51', lw=2, markersize=3)
    plt.title("Accuracy Curves - Custom DenseNet-121", fontweight="bold")
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    plt.tight_layout()
    curve_path = os.path.join(args.plots_dir, "densenet121_curves.png")
    plt.savefig(curve_path, dpi=300)
    plt.show()
    print(f">> Đã lưu biểu đồ: {curve_path}")
except Exception as e:
    print(f">> Bỏ qua vẽ biểu đồ ({e})")

# 2. Nạp lại trọng số của epoch tốt nhất (Best Acc) để đánh giá kiểm định TTA
if os.path.exists(best_ckpt):
    best_ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt_data["model_state_dict"])
    print(f">> Đã nạp trọng số tốt nhất: Epoch {best_ckpt_data['epoch']:02d} (Best Acc: {best_ckpt_data['best_acc']:.2f}%)")

model.eval()

# Đánh giá kiểm thử với 4-View Multi-Scale TTA
all_preds_raw, all_preds_tta, all_targets = [], [], []

with torch.no_grad():
    for inputs, targets, _ in testloader:
        inputs = inputs.to(device)
        targets_np = targets.cpu().numpy()

        # View 1: Ảnh gốc
        p1 = F.softmax(model(inputs), dim=1)
        preds_raw = p1.max(1)[1].cpu().numpy()

        # View 2: Lật ngang Horizontal Flip
        p2 = F.softmax(model(torch.flip(inputs, dims=[-1])), dim=1)

        # View 3: Cắt trọng tâm và phóng to 1.1x (Zoomed View)
        h, w = inputs.shape[2], inputs.shape[3]
        h_c, w_c = int(h * 0.9), int(w * 0.9)
        top, left = (h - h_c) // 2, (w - w_c) // 2
        crop = inputs[:, :, top:top+h_c, left:left+w_c]
        inputs_zoom = F.interpolate(crop, size=(h, w), mode="bilinear", align_corners=False)
        p3 = F.softmax(model(inputs_zoom), dim=1)

        # View 4: Phóng to kết hợp lật ngang
        p4 = F.softmax(model(torch.flip(inputs_zoom, dims=[-1])), dim=1)

        # Weighted Ensemble: 40% Gốc + 30% Flip + 15% Zoom + 15% Zoom-Flip
        probs_tta = (p1 * 0.4) + (p2 * 0.3) + (p3 * 0.15) + (p4 * 0.15)
        preds_tta = probs_tta.max(1)[1].cpu().numpy()

        all_preds_raw.extend(preds_raw)
        all_preds_tta.extend(preds_tta)
        all_targets.extend(targets_np)

all_preds_raw = np.array(all_preds_raw)
all_preds_tta = np.array(all_preds_tta)
all_targets = np.array(all_targets)

acc_raw = 100.0 * np.mean(all_preds_raw == all_targets)
acc_tta = 100.0 * np.mean(all_preds_tta == all_targets)

print()
print("================ KẾT QUẢ ĐỐI SÁNH ================")
print(f"1. Test Accuracy chuẩn (Không dùng TTA)     : {acc_raw:.2f}%")
print(f"2. Test Accuracy tối ưu với 4-View Multi-TTA : {acc_tta:.2f}%")
print(f">> Mức tăng trưởng đạt được nhờ 4-View TTA   : +{acc_tta - acc_raw:.2f}%")
print("===================================================")
print()

# 3. Vẽ Confusion Matrix chuẩn 37x37
try:
    if "classes" not in locals() or classes is None:
        if hasattr(testset, "class_names"):
            classes = testset.class_names
        else:
            classes = [f"Class_{c}" for c in range(37)]
            
    cm = confusion_matrix(all_targets, all_preds_tta)
    breed_names = [name.replace('_', ' ').title() for name in classes]

    plt.figure(figsize=(15, 13))
    im = plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f"MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - CUSTOM DENSENET-121 TRÊN 37 GIỐNG PETS", fontweight="bold", fontsize=12, pad=15)
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label('Số lượng ảnh', fontweight='bold', fontsize=11)

    tick_marks = np.arange(len(breed_names))
    plt.xticks(tick_marks, breed_names, rotation=90, fontsize=8)
    plt.yticks(tick_marks, breed_names, fontsize=8)

    plt.xlabel('Nhãn Dự Đoán (Predicted Breed)', fontweight='bold', fontsize=11, labelpad=10)
    plt.ylabel('Nhãn Thực Tế (Ground Truth Breed)', fontweight='bold', fontsize=11, labelpad=10)
    plt.tight_layout()
    cm_path = os.path.join(args.plots_dir, "densenet121_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f">> Đã lưu Confusion Matrix: {cm_path}")
except Exception as e:
    print(f">> Bỏ qua lưu Confusion Matrix ({e})")

# 4. Xuất file tóm tắt chỉ số định lượng
metrics_summary = {
    "model_name": "Custom DenseNet-121",
    "total_params": total_params,
    "batch_size": args.batch_size,
    "epochs_trained": len(train_losses),
    "final_train_loss": round(train_losses[-1], 3) if train_losses else None,
    "final_train_acc": round(train_accs[-1], 2) if train_accs else None,
    "final_test_loss": round(test_losses[-1], 3) if test_losses else None,
    "final_test_acc": round(test_accs[-1], 2) if test_accs else None,
    "best_test_acc_raw": round(acc_raw, 2),
    "best_test_acc_tta": round(acc_tta, 2),
    "delta_tta_gain": round(acc_tta - acc_raw, 2)
}
metrics_file = os.path.join(args.plots_dir, "densenet121_metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Đã lưu kết quả đối sánh tại: {metrics_file}")
